# 🌀 CycloneX — Worker GPU (Kaggle)

Roda este notebook como **Worker** conectado ao seu **Master** via Ngrok.

## ✅ Antes de começar:
1. No painel direito do Kaggle, ative **Accelerator → GPU T4 x2**
2. Ative **Internet → On** (obrigatório para clonar o repo)
3. Preencha **apenas a Célula 4** com a URL do Ngrok e o Token
4. Execute todas as células: **Run All**

> 💡 **Vantagem Kaggle:** 2x T4 GPUs = ~1 GKey/s por sessão!

## 🔧 Célula 1 — Verificar GPUs disponíveis

In [ ]:
import subprocess, os, sys, socket

print('=== GPU INFO ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('❌ GPU não detectada! Ative o acelerador GPU T4 x2 no painel direito.')
print(result.stdout)

# Detectar quantas GPUs estão disponíveis
cc_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,compute_cap,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpus = cc_result.stdout.strip().split('\n')
print(f'\n✅ {len(gpus)} GPU(s) disponível(is):')
for g in gpus:
    print(f'   {g.strip()}')

NUM_GPUS = len(gpus)
GPU_IDS  = ','.join(str(i) for i in range(NUM_GPUS))
print(f'\nGPU IDs para config: "{GPU_IDS}"')

## 📥 Célula 2 — Clonar repositório CycloneX

In [ ]:
REPO_DIR = '/kaggle/working/cycloneX'

if os.path.exists(REPO_DIR):
    print('Repositório já existe. Atualizando...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout)
else:
    print('Clonando repositório CycloneX...')
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/ggsofthouse/cycloneX.git', REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'❌ Clone falhou: {result.stderr}\n\nVerifique se Internet está ativada no painel direito!')

os.chdir(REPO_DIR)
print(f'\n✅ Repositório pronto em: {REPO_DIR}')
print('Arquivos:', [f for f in os.listdir('.') if not f.startswith('.')])

## ⚙️ Célula 3 — Compilar CUDACyclone para Linux

> ⏱️ Leva **2~4 minutos** na primeira vez. O Kaggle já tem `nvcc` instalado.

In [ ]:
CUDA_DIR = f'{REPO_DIR}/CUDACyclone-main'
BINARY   = f'{CUDA_DIR}/CUDACyclone'
SYMLINK  = f'{REPO_DIR}/CUDACyclone.exe'

if os.path.exists(BINARY):
    print(f'✅ Binário já compilado: {BINARY}')
else:
    # Garantir nvcc no PATH do Kaggle
    for cuda_path in ['/usr/local/cuda/bin', '/usr/local/cuda-12/bin', '/usr/local/cuda-11/bin']:
        if os.path.exists(f'{cuda_path}/nvcc'):
            os.environ['PATH'] = f"{cuda_path}:{os.environ.get('PATH', '')}"
            print(f'nvcc encontrado em: {cuda_path}')
            break

    # Detectar compute capability
    cc_raw = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True
    ).stdout.strip().split('\n')[0].replace('.', '')
    GPU_ARCH = cc_raw.strip() if cc_raw.strip() else '75'
    print(f'GPU Compute Capability: {GPU_ARCH}')
    print(f'\n🔨 Compilando CUDACyclone.cu (aguarde ~3 min)...\n')

    build = subprocess.run(
        ['make', f'GPU_ARCHS={GPU_ARCH}', '-j4'],
        capture_output=True, text=True,
        cwd=CUDA_DIR
    )
    output = (build.stdout + build.stderr)
    print(output[-3000:] if len(output) > 3000 else output)

    if build.returncode != 0 or not os.path.exists(BINARY):
        # Tentar com make clean primeiro
        subprocess.run(['make', 'clean'], cwd=CUDA_DIR, capture_output=True)
        build2 = subprocess.run(
            ['make', f'GPU_ARCHS={GPU_ARCH}', '-j4'],
            capture_output=True, text=True, cwd=CUDA_DIR
        )
        if build2.returncode != 0 or not os.path.exists(BINARY):
            raise RuntimeError('❌ Build falhou!\n' + build2.stderr[-1000:])

    print(f'\n✅ Build OK!')

# Symlink: cyclone_agent.py procura por "CUDACyclone.exe" — resolvido com symlink!
if os.path.islink(SYMLINK) or os.path.exists(SYMLINK):
    os.remove(SYMLINK)
os.symlink(BINARY, SYMLINK)

size = os.path.getsize(BINARY) / (1024*1024)
print(f'✅ Symlink: CUDACyclone.exe → {BINARY}')
print(f'   Tamanho: {size:.1f} MB')

## 🌐 Célula 4 — ⚠️ CONFIGURAR AQUI

> 🔒 Estas infos ficam **apenas nesta sessão** — não são salvas nem commitadas no GitHub.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  👇 EDITE APENAS ESTAS DUAS LINHAS:                 ║
# ╚══════════════════════════════════════════════════════╝

MASTER_URL = "https://lethargic-modulator-unworldly.ngrok-free.dev"  # ← URL do Ngrok do Master
TOKEN      = "cyclone_token_12345"                                    # ← Token do config.yaml

# ── Performance: Kaggle usa 2x T4, então aproveitamos as 2 GPUs ──
GRID   = "512,1024"
SLICES = 256
# GPU_IDS é detectado automaticamente na Célula 1

# ────────────────────────────────────────────────────────────────
print(f'Master URL : {MASTER_URL}')
print(f'Token      : {TOKEN[:4]}...{TOKEN[-4:]}  (parcialmente oculto)')
print(f'Grid       : {GRID}')
print(f'Slices     : {SLICES}')
print(f'GPUs       : {GPU_IDS}  ({NUM_GPUS} GPU(s))')

## 📝 Célula 5 — Gerar config.yaml (modo Worker)

In [ ]:
os.chdir(REPO_DIR)

worker_id = socket.gethostname()
print(f'Worker ID: {worker_id}')

config_yaml = f"""job:
  grid: \"{GRID}\"
  slices: {SLICES}
  gpus: \"{GPU_IDS}\"
  random: false

gpu:
  max_temp_c: 85

telemetry:
  interval: 1

api:
  port: 8081
  token: \"{TOKEN}\"

database:
  path: \"./cyclone_worker.db\"

pool:
  role: \"worker\"
  master_url: \"{MASTER_URL}\"
  block_size_gkeys: 200
  random_in_block: false
"""

with open('config.yaml', 'w') as f:
    f.write(config_yaml)

print('\n✅ config.yaml gerado:')
print(config_yaml)

## 🔗 Célula 6 — Testar conexão com o Master

In [ ]:
import urllib.request, json, time

print(f'Testando conexão com {MASTER_URL} ...')

for attempt in range(3):
    try:
        req = urllib.request.Request(
            f'{MASTER_URL}/api/status',
            headers={
                'Authorization': f'Bearer {TOKEN}',
                'ngrok-skip-browser-warning': 'true'
            }
        )
        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.loads(r.read())
        print(f'\n✅ Master acessível!')
        print(f'   Job rodando : {data.get("job_running", "?")}')
        print(f'   Speed       : {data.get("speed_mkeys", 0):.1f} MKeys/s')
        print(f'   Workers     : {len(data.get("pool_workers", {}))} conectados')
        break
    except Exception as e:
        print(f'Tentativa {attempt+1}/3 falhou: {e}')
        if attempt < 2:
            time.sleep(5)
        else:
            print('\n❌ Não foi possível conectar ao Master.')
            print('Verifique se o ngrok e o cyclone_agent.py estão rodando no Master!')

## 🚀 Célula 7 — Iniciar Worker CycloneX

> Esta célula roda até o Kaggle encerrar a sessão (~12h).
>
> Acompanhe no dashboard do Master: `http://localhost:8080` → aba **Pool**

In [ ]:
os.chdir(REPO_DIR)

print('🚀 Iniciando CycloneX Worker (Kaggle)...')
print(f'   Master  : {MASTER_URL}')
print(f'   Worker  : {socket.gethostname()}')
print(f'   GPUs    : {NUM_GPUS}x T4')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, 'cyclone_agent.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)

try:
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('\n⛔ Worker encerrado.')
finally:
    if proc.poll() is None:
        proc.terminate()

---

## 📊 Referência rápida

| GPU Kaggle | Velocidade estimada |
|------------|--------------------|
| T4 x1 | ~500 MKeys/s |
| **T4 x2** | **~1 GKey/s** 🔥 |
| P100 | ~800 MKeys/s |

## ⚙️ Como usar as 2 GPUs

O `config.yaml` já foi gerado com `gpus: "0,1"` automaticamente.
O CUDACyclone divide o trabalho entre as duas GPUs!

---
*CycloneX v2.0 — Distributed GPU Worker (Kaggle Edition)*